<a href="https://colab.research.google.com/github/Bat4E/Csci164-Colab/blob/main/Search164.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import random
import heapq
import time
import copy
from collections import deque

# Problem class and Node definitions (same as original)
class Problem(object): pass

class Node(object):
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.State = state
        self.Parent = parent
        self.Action = action
        self.PathCost = path_cost

    def __str__(self):
        action = "<none>" if not self.Action else self.Action
        return str(self.State) + ", " + action

    def __repr__(self):
        action = "<none>" if not self.Action else self.Action
        return str(self.State) + ", " + action

    def __lt__(self, other):
        return self.PathCost < other.PathCost

# 3x3 Puzzle Constants
StateDimension3 = 3
InitialState3 = [1, 2, 3, 4, 5, 6, 7, 8, 0]
GoalState3 = [1, 2, 3, 4, 5, 6, 7, 8, 0]

# 4x4 Puzzle Constants (new addition)
StateDimension4 = 4
InitialState4 = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
GoalState4 = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]

# Common functions for both puzzle sizes
Actions = lambda s: ['u', 'd', 'l', 'r']
Opposite = dict([('u', 'd'), ('d', 'u'), ('l', 'r'), ('r', 'l'), (None, None)])

def Result3(state, action):
    """Calculate the result of an action on a 3x3 puzzle state"""
    i = state.index(0)
    newState = list(state)
    row, col = i // StateDimension3, i % StateDimension3

    # Check if move is valid
    if ((action == 'u' and row == 0) or
        (action == 'd' and row == StateDimension3 - 1) or
        (action == 'l' and col == 0) or
        (action == 'r' and col == StateDimension3 - 1)):
        return newState

    # Calculate indices for swap
    if action == 'u':
        l, r = row * StateDimension3 + col, (row - 1) * StateDimension3 + col
    elif action == 'd':
        l, r = row * StateDimension3 + col, (row + 1) * StateDimension3 + col
    elif action == 'l':
        l, r = row * StateDimension3 + col, row * StateDimension3 + col - 1
    elif action == 'r':
        l, r = row * StateDimension3 + col, row * StateDimension3 + col + 1

    # Swap the blank with the adjacent tile
    newState[l], newState[r] = newState[r], newState[l]
    return newState

def Result4(state, action):
    """Calculate the result of an action on a 4x4 puzzle state"""
    i = state.index(0)
    newState = list(state)
    row, col = i // StateDimension4, i % StateDimension4

    # Check if move is valid
    if ((action == 'u' and row == 0) or
        (action == 'd' and row == StateDimension4 - 1) or
        (action == 'l' and col == 0) or
        (action == 'r' and col == StateDimension4 - 1)):
        return newState

    # Calculate indices for swap
    if action == 'u':
        l, r = row * StateDimension4 + col, (row - 1) * StateDimension4 + col
    elif action == 'd':
        l, r = row * StateDimension4 + col, (row + 1) * StateDimension4 + col
    elif action == 'l':
        l, r = row * StateDimension4 + col, row * StateDimension4 + col - 1
    elif action == 'r':
        l, r = row * StateDimension4 + col, row * StateDimension4 + col + 1

    # Swap the blank with the adjacent tile
    newState[l], newState[r] = newState[r], newState[l]
    return newState

def LegalMove3(state, action):
    """Check if a move is legal for 3x3 puzzle"""
    i = state.index(0)
    row, col = i // StateDimension3, i % StateDimension3

    if ((action == 'u' and row == 0) or
        (action == 'd' and row == StateDimension3 - 1) or
        (action == 'l' and col == 0) or
        (action == 'r' and col == StateDimension3 - 1)):
        return False
    return True

def LegalMove4(state, action):
    """Check if a move is legal for 4x4 puzzle"""
    i = state.index(0)
    row, col = i // StateDimension4, i % StateDimension4

    if ((action == 'u' and row == 0) or
        (action == 'd' and row == StateDimension4 - 1) or
        (action == 'l' and col == 0) or
        (action == 'r' and col == StateDimension4 - 1)):
        return False
    return True

def PrintState(s, dimension=3):
    """Print a state in a readable format"""
    for i in range(0, len(s), dimension):
        print(s[i:i+dimension])

def RandomWalk3(state, steps):
    """Perform a random walk from a state for a 3x3 puzzle"""
    actionSequence = []
    actionLast = None
    currentState = list(state)

    for i in range(steps):
        action = None
        while action == None:
            action = random.choice(Actions(state))
            action = action if (LegalMove3(currentState, action) and
                               action != Opposite[actionLast]) else None
        actionLast = action
        currentState = Result3(currentState, action)
        actionSequence.append(action)

    return currentState, actionSequence

def RandomWalk4(state, steps):
    """Perform a random walk from a state for a 4x4 puzzle"""
    actionSequence = []
    actionLast = None
    currentState = list(state)

    for i in range(steps):
        action = None
        while action == None:
            action = random.choice(Actions(state))
            action = action if (LegalMove4(currentState, action) and
                               action != Opposite[actionLast]) else None
        actionLast = action
        currentState = Result4(currentState, action)
        actionSequence.append(action)

    return currentState, actionSequence

def ApplyMoves(actions, state, size=3):
    """Apply a sequence of moves to a state"""
    current_state = list(state)
    for action in actions:
        if size == 3:
            current_state = Result3(current_state, action)
        else:
            current_state = Result4(current_state, action)
    return current_state

def Solution(node):
    """Reconstruct the path from the initial state to this state"""
    if node.Parent == None:
        return []
    return Solution(node.Parent) + [node.Action]

def Expand(problem, node):
    """Expand a node by generating all possible successor states"""
    ret = []
    s = node.State
    for action in problem.Actions(s):
        sPrime = problem.Result(s, action)
        cost = node.PathCost + problem.ActionCost(s, action, sPrime)
        ret.append(Node(sPrime, node, action, cost))
    return ret

# Heuristic functions
def SingleTileManhattanDistance3(tile, left, right):
    """Calculate Manhattan distance for a single tile in a 3x3 puzzle"""
    leftIndex = left.index(tile)
    rightIndex = right.index(tile)
    return (abs(leftIndex // StateDimension3 - rightIndex // StateDimension3) +
            abs(leftIndex % StateDimension3 - rightIndex % StateDimension3))

def ManhattanDistance3(left, right):
    """Calculate total Manhattan distance for a 3x3 puzzle"""
    distances = [SingleTileManhattanDistance3(tile, left, right)
                for tile in range(1, StateDimension3**2)]
    return sum(distances)

def SingleTileManhattanDistance4(tile, left, right):
    """Calculate Manhattan distance for a single tile in a 4x4 puzzle"""
    leftIndex = left.index(tile)
    rightIndex = right.index(tile)
    return (abs(leftIndex // StateDimension4 - rightIndex // StateDimension4) +
            abs(leftIndex % StateDimension4 - rightIndex % StateDimension4))

def ManhattanDistance4(left, right):
    """Calculate total Manhattan distance for a 4x4 puzzle"""
    distances = [SingleTileManhattanDistance4(tile, left, right)
                for tile in range(1, StateDimension4**2)]
    return sum(distances)

def OutOfPlace3(left, right):
    """Calculate number of tiles out of place for a 3x3 puzzle"""
    distances = [left[i] != right[i] and right[i] != 0
                for i in range(StateDimension3**2)]
    return sum(distances)

def OutOfPlace4(left, right):
    """Calculate number of tiles out of place for a 4x4 puzzle"""
    distances = [left[i] != right[i] and right[i] != 0
                for i in range(StateDimension4**2)]
    return sum(distances)

# Search algorithms
def BreadthFirstSearch(problem):
    """Breadth-First Search algorithm"""
    node = Node(list(problem.INITIAL))
    if problem.IsGoal(node.State):
        return node, 0

    frontier = deque([node])
    reached = set()
    reached.add(tuple(problem.INITIAL))
    nodesExpanded = 0

    while frontier:
        node = frontier.popleft()

        for child in Expand(problem, node):
            s = tuple(child.State)
            if problem.IsGoal(s):
                return child, nodesExpanded
            if s not in reached:
                reached.add(s)
                frontier.append(child)

        nodesExpanded += 1
        if nodesExpanded > 500000:  # Safety limit
            break

    return None, nodesExpanded

def BestFirstSearch(problem, f):
    """Best-First Search algorithm (used for A* implementation)"""
    node = Node(list(problem.INITIAL))

    frontier = []
    heapq.heappush(frontier, (f(node), id(node), node))
    reached = {}
    reached[tuple(problem.INITIAL)] = node
    nodesExpanded = 0

    while frontier:
        _, _, node = heapq.heappop(frontier)

        if problem.IsGoal(tuple(node.State)):
            return node, nodesExpanded

        for child in Expand(problem, node):
            s = tuple(child.State)
            if s not in reached or child.PathCost < reached[s].PathCost:
                reached[s] = child
                heapq.heappush(frontier, (f(child), id(child), child))

        nodesExpanded += 1
        if nodesExpanded > 500000:  # Safety limit
            break

    return None, nodesExpanded

# Experiment functions
def run_experiments():
    """Run experiments as specified in the assignment"""
    # Configure problems for 3x3 and 4x4 puzzles
    puzzle3 = Problem()
    puzzle3.INITIAL = InitialState3
    puzzle3.GoalState = GoalState3
    puzzle3.IsGoal = lambda s: tuple(s) == tuple(GoalState3)
    puzzle3.Actions = Actions
    puzzle3.Result = Result3
    puzzle3.ActionCost = lambda s, a, sPrime: 1

    puzzle4 = Problem()
    puzzle4.INITIAL = InitialState4
    puzzle4.GoalState = GoalState4
    puzzle4.IsGoal = lambda s: tuple(s) == tuple(GoalState4)
    puzzle4.Actions = Actions
    puzzle4.Result = Result4
    puzzle4.ActionCost = lambda s, a, sPrime: 1

    # Heuristic functions for A*
    manhattanF3 = lambda n: n.PathCost + ManhattanDistance3(n.State, GoalState3)
    outOfPlaceF3 = lambda n: n.PathCost + OutOfPlace3(n.State, GoalState3)
    manhattanF4 = lambda n: n.PathCost + ManhattanDistance4(n.State, GoalState4)
    outOfPlaceF4 = lambda n: n.PathCost + OutOfPlace4(n.State, GoalState4)

    # Step counts as specified in the assignment
    step_counts = [5, 10, 20, 40, 80]

    # Results storage
    results3 = []
    results4 = []

    # Set random seed for reproducibility
    random.seed(123456780)

    # Generate 3 problems for each step count for 3x3 puzzle
    print("\n=== 3x3 Puzzle Experiments ===")
    for steps in step_counts:
        for i in range(3):  # 3 problems per step count
            print(f"\nProblem {i+1} with {steps} random steps:")

            # Generate random problem
            initial_state, _ = RandomWalk3(GoalState3, steps)
            print("Initial state:")
            PrintState(initial_state, 3)

            # Store problem data
            problem_data = {
                "steps": steps,
                "initial_state": list(initial_state),
                "bfs": {},
                "a_star_manhattan": {},
                "a_star_outofplace": {}
            }

            # Test BFS
            print("\nTesting BFS...")
            puzzle3.INITIAL = initial_state
            start_time = time.time()
            node_bfs, nodes_expanded = BreadthFirstSearch(puzzle3)
            time_taken = time.time() - start_time

            if node_bfs:
                solution = Solution(node_bfs)
                print(f"Solution found: {solution}")
                print(f"Solution length: {len(solution)}")
                print(f"Nodes expanded: {nodes_expanded}")

                problem_data["bfs"] = {
                    "solution": solution,
                    "solution_length": len(solution),
                    "nodes_expanded": nodes_expanded,
                    "time_taken": time_taken
                }
            else:
                print("No solution found")
                problem_data["bfs"] = {
                    "solution": None,
                    "solution_length": None,
                    "nodes_expanded": nodes_expanded,
                    "time_taken": time_taken
                }

            # Test A* with Manhattan distance
            print("\nTesting A* with Manhattan distance...")
            puzzle3.INITIAL = initial_state
            start_time = time.time()
            node_astar_man, nodes_expanded = BestFirstSearch(puzzle3, manhattanF3)
            time_taken = time.time() - start_time

            if node_astar_man:
                solution = Solution(node_astar_man)
                print(f"Solution found: {solution}")
                print(f"Solution length: {len(solution)}")
                print(f"Nodes expanded: {nodes_expanded}")

                problem_data["a_star_manhattan"] = {
                    "solution": solution,
                    "solution_length": len(solution),
                    "nodes_expanded": nodes_expanded,
                    "time_taken": time_taken
                }
            else:
                print("No solution found")
                problem_data["a_star_manhattan"] = {
                    "solution": None,
                    "solution_length": None,
                    "nodes_expanded": nodes_expanded,
                    "time_taken": time_taken
                }

            # Test A* with Out-of-place
            print("\nTesting A* with Out-of-place...")
            puzzle3.INITIAL = initial_state
            start_time = time.time()
            node_astar_oop, nodes_expanded = BestFirstSearch(puzzle3, outOfPlaceF3)
            time_taken = time.time() - start_time

            if node_astar_oop:
                solution = Solution(node_astar_oop)
                print(f"Solution found: {solution}")
                print(f"Solution length: {len(solution)}")
                print(f"Nodes expanded: {nodes_expanded}")

                problem_data["a_star_outofplace"] = {
                    "solution": solution,
                    "solution_length": len(solution),
                    "nodes_expanded": nodes_expanded,
                    "time_taken": time_taken
                }
            else:
                print("No solution found")
                problem_data["a_star_outofplace"] = {
                    "solution": None,
                    "solution_length": None,
                    "nodes_expanded": nodes_expanded,
                    "time_taken": time_taken
                }

            # Add problem data to results
            results3.append(problem_data)

    # Generate 3 problems for each step count for 4x4 puzzle
    print("\n=== 4x4 Puzzle Experiments ===")
    for steps in step_counts:
        for i in range(3):  # 3 problems per step count
            print(f"\nProblem {i+1} with {steps} random steps:")

            # Generate random problem
            initial_state, _ = RandomWalk4(GoalState4, steps)
            print("Initial state:")
            PrintState(initial_state, 4)

            # Store problem data
            problem_data = {
                "steps": steps,
                "initial_state": list(initial_state),
                "bfs": {},
                "a_star_manhattan": {},
                "a_star_outofplace": {}
            }

            # Test BFS
            print("\nTesting BFS...")
            puzzle4.INITIAL = initial_state
            start_time = time.time()
            node_bfs, nodes_expanded = BreadthFirstSearch(puzzle4)
            time_taken = time.time() - start_time

            if node_bfs:
                solution = Solution(node_bfs)
                print(f"Solution found: {solution}")
                print(f"Solution length: {len(solution)}")
                print(f"Nodes expanded: {nodes_expanded}")

                problem_data["bfs"] = {
                    "solution": solution,
                    "solution_length": len(solution),
                    "nodes_expanded": nodes_expanded,
                    "time_taken": time_taken
                }
            else:
                print("No solution found")
                problem_data["bfs"] = {
                    "solution": None,
                    "solution_length": None,
                    "nodes_expanded": nodes_expanded,
                    "time_taken": time_taken
                }

            # Test A* with Manhattan distance
            print("\nTesting A* with Manhattan distance...")
            puzzle4.INITIAL = initial_state
            start_time = time.time()
            node_astar_man, nodes_expanded = BestFirstSearch(puzzle4, manhattanF4)
            time_taken = time.time() - start_time

            if node_astar_man:
                solution = Solution(node_astar_man)
                print(f"Solution found: {solution}")
                print(f"Solution length: {len(solution)}")
                print(f"Nodes expanded: {nodes_expanded}")

                problem_data["a_star_manhattan"] = {
                    "solution": solution,
                    "solution_length": len(solution),
                    "nodes_expanded": nodes_expanded,
                    "time_taken": time_taken
                }
            else:
                print("No solution found")
                problem_data["a_star_manhattan"] = {
                    "solution": None,
                    "solution_length": None,
                    "nodes_expanded": nodes_expanded,
                    "time_taken": time_taken
                }

            # Test A* with Out-of-place
            print("\nTesting A* with Out-of-place...")
            puzzle4.INITIAL = initial_state
            start_time = time.time()
            node_astar_oop, nodes_expanded = BestFirstSearch(puzzle4, outOfPlaceF4)
            time_taken = time.time() - start_time

            if node_astar_oop:
                solution = Solution(node_astar_oop)
                print(f"Solution found: {solution}")
                print(f"Solution length: {len(solution)}")
                print(f"Nodes expanded: {nodes_expanded}")

                problem_data["a_star_outofplace"] = {
                    "solution": solution,
                    "solution_length": len(solution),
                    "nodes_expanded": nodes_expanded,
                    "time_taken": time_taken
                }
            else:
                print("No solution found")
                problem_data["a_star_outofplace"] = {
                    "solution": None,
                    "solution_length": None,
                    "nodes_expanded": nodes_expanded,
                    "time_taken": time_taken
                }

            # Add problem data to results
            results4.append(problem_data)

    return results3, results4

def summary_and_analysis(results3, results4):
    """Summarize results and provide analysis"""
    print("\n=== Summary of Results ===")

    # 3x3 Puzzle Summary
    print("\n3x3 Puzzle Summary:")
    for steps in [5, 10, 20, 40, 80]:
        step_results = [r for r in results3 if r["steps"] == steps]

        if not step_results:
            continue

        print(f"\n  Random Walk Steps: {steps}")

        # BFS stats
        bfs_nodes = [r["bfs"].get("nodes_expanded") for r in step_results if r["bfs"].get("nodes_expanded") is not None]
        bfs_lengths = [r["bfs"].get("solution_length") for r in step_results if r["bfs"].get("solution_length") is not None]

        if bfs_nodes and bfs_lengths:
            print(f"  BFS Average Nodes Expanded: {sum(bfs_nodes) / len(bfs_nodes):.1f}")
            print(f"  BFS Average Solution Length: {sum(bfs_lengths) / len(bfs_lengths):.1f}")

        # A* Manhattan stats
        astar_m_nodes = [r["a_star_manhattan"].get("nodes_expanded") for r in step_results if r["a_star_manhattan"].get("nodes_expanded") is not None]
        astar_m_lengths = [r["a_star_manhattan"].get("solution_length") for r in step_results if r["a_star_manhattan"].get("solution_length") is not None]

        if astar_m_nodes and astar_m_lengths:
            print(f"  A* Manhattan Average Nodes Expanded: {sum(astar_m_nodes) / len(astar_m_nodes):.1f}")
            print(f"  A* Manhattan Average Solution Length: {sum(astar_m_lengths) / len(astar_m_lengths):.1f}")

        # A* Out-of-place stats
        astar_o_nodes = [r["a_star_outofplace"].get("nodes_expanded") for r in step_results if r["a_star_outofplace"].get("nodes_expanded") is not None]
        astar_o_lengths = [r["a_star_outofplace"].get("solution_length") for r in step_results if r["a_star_outofplace"].get("solution_length") is not None]

        if astar_o_nodes and astar_o_lengths:
            print(f"  A* Out-of-place Average Nodes Expanded: {sum(astar_o_nodes) / len(astar_o_nodes):.1f}")
            print(f"  A* Out-of-place Average Solution Length: {sum(astar_o_lengths) / len(astar_o_lengths):.1f}")

    # 4x4 Puzzle Summary
    print("\n4x4 Puzzle Summary:")
    for steps in [5, 10, 20, 40, 80]:
        step_results = [r for r in results4 if r["steps"] == steps]

        if not step_results:
            continue

        print(f"\n  Random Walk Steps: {steps}")

        # BFS stats
        bfs_nodes = [r["bfs"].get("nodes_expanded") for r in step_results if r["bfs"].get("nodes_expanded") is not None]
        bfs_lengths = [r["bfs"].get("solution_length") for r in step_results if r["bfs"].get("solution_length") is not None]

        if bfs_nodes:
            print(f"  BFS Average Nodes Expanded: {sum(bfs_nodes) / len(bfs_nodes):.1f}")

            if bfs_lengths:
                print(f"  BFS Average Solution Length: {sum(bfs_lengths) / len(bfs_lengths):.1f}")
            else:
                print("  BFS: No solutions found")

        # A* Manhattan stats
        astar_m_nodes = [r["a_star_manhattan"].get("nodes_expanded") for r in step_results if r["a_star_manhattan"].get("nodes_expanded") is not None]
        astar_m_lengths = [r["a_star_manhattan"].get("solution_length") for r in step_results if r["a_star_manhattan"].get("solution_length") is not None]

        if astar_m_nodes:
            print(f"  A* Manhattan Average Nodes Expanded: {sum(astar_m_nodes) / len(astar_m_nodes):.1f}")

            if astar_m_lengths:
                print(f"  A* Manhattan Average Solution Length: {sum(astar_m_lengths) / len(astar_m_lengths):.1f}")
            else:
                print("  A* Manhattan: No solutions found")

        # A* Out-of-place stats
        astar_o_nodes = [r["a_star_outofplace"].get("nodes_expanded") for r in step_results if r["a_star_outofplace"].get("nodes_expanded") is not None]
        astar_o_lengths = [r["a_star_outofplace"].get("solution_length") for r in step_results if r["a_star_outofplace"].get("solution_length") is not None]

        if astar_o_nodes:
            print(f"  A* Out-of-place Average Nodes Expanded: {sum(astar_o_nodes) / len(astar_o_nodes):.1f}")

            if astar_o_lengths:
                print(f"  A* Out-of-place Average Solution Length: {sum(astar_o_lengths) / len(astar_o_lengths):.1f}")
            else:
                print("  A* Out-of-place: No solutions found")

    # Analysis
    print("\n=== Analysis ===")
    print("\nSearch as a Tool for Artificial Intelligence:")
    print("1. State Space Complexity: The 4x4 puzzle has a significantly larger state space (10^13 states) compared to the 3x3 puzzle (10^5 states). This is evident from our experiments where algorithms struggle more with 4x4 puzzles.")
    print("2. Informed vs. Uninformed Search: A* consistently outperforms BFS, especially on more complex problems, demonstrating the value of heuristic guidance in AI problem solving.")
    print("3. Heuristic Quality: Manhattan distance provides better guidance than out-of-place, resulting in fewer nodes expanded while still finding optimal solutions.")
    print("4. Problem Size Impact: As problem complexity increases, the performance gap between informed and uninformed search widens, highlighting the importance of effective heuristics for complex AI problems.")
    print("5. Problem Difficulty: The number of random steps correlates with problem difficulty, but not perfectly. Some 'shorter' random walks can still create difficult puzzles.")
    print("6. Real-world AI Applications: These findings extend to many AI domains including planning, pathfinding, and optimization where informed search algorithms and effective heuristics are essential for handling large state spaces.")

# Main
if __name__ == "__main__":
    # Run experiments
    results3, results4 = run_experiments()

    # Summarize and analyze results
    summary_and_analysis(results3, results4)


=== 3x3 Puzzle Experiments ===

Problem 1 with 5 random steps:
Initial state:
[1, 3, 6]
[4, 2, 0]
[7, 5, 8]

Testing BFS...
Solution found: ['u', 'l', 'd', 'd', 'r']
Solution length: 5
Nodes expanded: 19

Testing A* with Manhattan distance...
Solution found: ['u', 'l', 'd', 'd', 'r']
Solution length: 5
Nodes expanded: 5

Testing A* with Out-of-place...
Solution found: ['u', 'l', 'd', 'd', 'r']
Solution length: 5
Nodes expanded: 5

Problem 2 with 5 random steps:
Initial state:
[4, 1, 2]
[0, 5, 3]
[7, 8, 6]

Testing BFS...
Solution found: ['u', 'r', 'r', 'd', 'd']
Solution length: 5
Nodes expanded: 22

Testing A* with Manhattan distance...
Solution found: ['u', 'r', 'r', 'd', 'd']
Solution length: 5
Nodes expanded: 5

Testing A* with Out-of-place...
Solution found: ['u', 'r', 'r', 'd', 'd']
Solution length: 5
Nodes expanded: 5

Problem 3 with 5 random steps:
Initial state:
[1, 5, 2]
[0, 4, 3]
[7, 8, 6]

Testing BFS...
Solution found: ['r', 'u', 'r', 'd', 'd']
Solution length: 5
Nodes ex